# Figure 5 — TCR–pMHC binding benchmark (figure + tables + significance)

Reproduces manuscript **Figure 5** (combined a/b/c) and its result tables from the stored tables in `../data/`, then runs the paired-Wilcoxon significance tests with Holm's correction. Run from `publication/notebooks/`.

**Outputs** — `../figures/fig5/figure5_benchmark.{svg,png}`, `../results/{delta_auc_table.tsv, figure5_panelA_auc_table.csv, significance_tests.csv, significance_power_summary.csv, iggytop_unseen_conclusive.csv}`.



In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import glob
import seaborn as sns
from sklearn.metrics import roc_auc_score
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

import sys
from pathlib import Path
PUB = Path.cwd().parent
sys.path.insert(0, str(PUB / "lib"))
from benchmarker import Benchmarker
bmarker = Benchmarker()

# Self-contained repo paths (no absolute filesystem paths in this notebook).
DATA, RESULTS, FIGURES = PUB / "data", PUB / "results", PUB / "figures"

In [ ]:
# Figure styling: editable-SVG text, white background, 350 dpi, >=12 pt fonts.
from figure_style import apply_style, figsize, set_y_headroom, add_panel_labels, save_figure

flierprops = apply_style()

In [ ]:
# read in the dataset
dataset = str(DATA / "stcrdab_calibrated")

predictor_results = glob.glob(f"{dataset}/*")

predictor_results

In [ ]:
def read_table(f_p):
    if f_p.endswith("csv"):
        return pd.read_csv(f_p)
    elif f_p.endswith("tsv"):
        return pd.read_csv(f_p, sep="\t")
    else: 
        return "Incorrect file ending. Needs to be 'tsv' or 'csv'"

def get_train_df(model):
        if model.lower() in ["gcn", "gat", "gcn-globmean", "gcn-100"]:
            return read_table(str(DATA / "training_data" / "t2pmhc_train_core.tsv"))
        elif model.lower() in ["mixtcrpred", "mixtcrpred-pan"]:
            return read_table(str(DATA / "training_data" / "mixtcrpred_full_training_set_146pmhc.csv"))
        else:
            raise ValueError(f"Train df -- Unsupported model: {model}")

def get_seen(model):
    train_df = get_train_df(model)
    if "peptide" in train_df.columns:
        return set(train_df["peptide"])
    elif "Peptide" in train_df.columns:
        return set(train_df["Peptide"])
    else: 
        return set(train_df["epitope"])

def get_unseen(f_p, model):
    seen_peptides = get_seen(model)
    return set(read_table(f_p)["peptide"]) - seen_peptides

In [ ]:
# lets split into seen/unseen here

# get seen/unseen
def get_shared_peptides(models, paths, seen, bmarker):


    if seen:
        peptide_sets = [get_seen(model) for model, f_p in zip(models, paths) if os.path.exists(f_p)]
    else:
        peptide_sets = [get_unseen(f_p, model) for model, f_p in zip(models, paths) if os.path.exists(f_p)]

    return set.intersection(*peptide_sets)

In [ ]:
def plot_violins_combined(df, ax):
    """
    Publication-ready violin plot.
    - Axis-aware
    - Does NOT touch model ordering
    - No per-axis labels (for shared fig labels)
    """
    df = df.copy()

    # Define model order and get color palette
    model_order = ['gcn', 'gat', 'mixtcrpred-pan']
    # Filter to only models present in data
    model_order = [m for m in model_order if m in df['model'].unique()]
    # gcn / gat keep their model colours; mixtcrpred-pan is a baseline, so it gets the
    # same baseline grey (BASELINE_DOT) used for the diamonds/boxes in panels (a) and (b).
    palette = {m: ("#808080" if m == "mixtcrpred-pan" else bmarker.get_color(m))
               for m in model_order}

    sns.violinplot(
        data=df,
        x='model',
        y='binder_prob_calibrated',
        order=model_order,
        palette=palette,
        inner='box',
        linewidth=1.5,
        saturation=0.8,
        cut=0,
        ax=ax
    )

    # Remove per-axis labels (shared labels at figure level)
    ax.set_xlabel("")
    ax.set_ylabel("")

    # Y-axis formatting (shared across panels): 0-1 in 0.2 steps
    ax.set_ylim(0, 1)
    ax.set_yticks(np.arange(0, 1.01, 0.2))

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=12,
        width=1.5,
        length=6
    )
    ax.tick_params(
        axis='both',
        which='minor',
        width=1,
        length=3
    )

    ax.set_axisbelow(True)

    # Publication spines
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(1.5)
        ax.spines[spine].set_color('#333333')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

In [ ]:
seen_peptides = get_shared_peptides(["gcn", "mixtcrpred"], [f"{dataset}/gcn_calibrated.tsv",f"{dataset}/mixtcrpred-pan_calibrated.tsv"], True, bmarker)

seen_df_list = list()

print("================")
print("SEEN")
print("================")

for file_path in glob.glob(f"{dataset}/*"):
    if not file_path.endswith("versions.yml"):
        print(file_path)
        model = bmarker.get_model(file_path)
        if model not in ["gcn", "gat", "mixtcrpred-pan"]:
            continue
        df = bmarker.read_table(file_path)
        df["model"] = model
        # as I forgot to do before, lets split into seen/unseen here
        # seen
        seen_df = df[df["peptide"].isin(seen_peptides)]
        # plot
        seen_df_list.append(seen_df)

combined_seen_df = pd.concat(seen_df_list, ignore_index=True)


print("================")
print("UNSEEN")
print("================")

# get unseen peptides for dataset 
unseen_peptides = get_shared_peptides(["gcn", "mixtcrpred"], [f"{dataset}/gcn_calibrated.tsv",f"{dataset}/mixtcrpred-pan_calibrated.tsv"], False, bmarker)

unseen_df_list = list()


for file_path in glob.glob(f"{dataset}/*"):
    if not file_path.endswith("versions.yml"):
        print(file_path)
        model = bmarker.get_model(file_path)
        if model not in ["gcn", "gat", "mixtcrpred-pan"]:
            continue
        df = bmarker.read_table(file_path)
        df["model"] = model
        # as I forgot to do before, lets split into seen/unseen here
        # unseen
        unseen_df = df[df["peptide"].isin(unseen_peptides)]
        # plot
        prob_col = "binder_prob_calibrated"
        unseen_df_list.append(unseen_df)

combined_unseen_df = pd.concat(unseen_df_list, ignore_index=True)

In [ ]:
# === Table 2: stCRDab calibrated binding probability, per model x Seen/Unseen (per-structure) ===
# Reported in the manuscript: N, mean, SD (ddof=1), median, % > 0.9, % < 0.5 of
# binder_prob_calibrated, on the shared seen/unseen split (combined_seen_df / combined_unseen_df).
_T2_MODELS = ["gcn", "gat", "mixtcrpred-pan"]
_T2_DISP = {"gcn": "t2pmhc-GCN", "gat": "t2pmhc-GAT", "mixtcrpred-pan": "MixTCRpred-pan"}
_t2_rows = []
for _split, _df in [("Seen", combined_seen_df), ("Unseen", combined_unseen_df)]:
    for _m in _T2_MODELS:
        v = _df[_df["model"] == _m]["binder_prob_calibrated"].to_numpy()
        _t2_rows.append({
            "model": _T2_DISP[_m], "split": _split, "N": len(v),
            "mean": v.mean(), "sd": v.std(ddof=1), "median": float(np.median(v)),
            "mean_sd": f"{v.mean():.2f} ± {v.std(ddof=1):.2f}",
            "pct_gt_0.9": 100 * np.mean(v > 0.9), "pct_lt_0.5": 100 * np.mean(v < 0.5),
        })
table2 = pd.DataFrame(_t2_rows)
table2.to_csv(RESULTS / "figure5_table2_stcrdab_probabilities.csv", index=False)
print(f"Table 2 ({len(table2)} rows) -> {RESULTS / 'figure5_table2_stcrdab_probabilities.csv'}")
table2.round(3)

In [ ]:
# ============================================================================
# Panels (a) + (b) setup -- analyzer paths, palettes, loader
# ============================================================================
ANALYZER_DIR_MAIN  = str(DATA / "analyzer")
ANALYZER_DIR_TCREN = str(DATA / "analyzer_tcren")
MPREDPAN_DIR = str(DATA / "mixtcrpred_pan")
PAE_DIR = str(DATA / "tcrdock_pae")

DATASETS = ["public_set", "immrep23", "epytope_viral", "iggytop_unseen"]

# bmarker palette lacks tcren / tulip-tcr / panpep; supply our own.
ML_EXTRA_COLORS = {"tcren": "#5b5c5b", "tulip-tcr": "#656367", "panpep": "#909090"}
TCRDOCK_COLORS  = {"tcrdock_pae": "#4d4d4d"}
PALETTE_BASE = {**bmarker.get_color_palette(), **ML_EXTRA_COLORS, **TCRDOCK_COLORS}

INVERTED_MODELS = ["tcren"]
_TCREN_COLS = ["binding_score_tcren", "sample_in_train_tcren", "seen_in_tcren"]


def _load_analyzer(ds):
    """Main analyzer table for `ds` + merged tcren columns + mixtcrpred-pan scores."""
    df = pd.read_table(f"{ANALYZER_DIR_MAIN}/{ds}_analyzer_table.tsv").drop_duplicates("identifier")
    tcren = pd.read_table(
        f"{ANALYZER_DIR_TCREN}/{ds}_analyzer_table.tsv",
        usecols=["identifier"] + _TCREN_COLS,
    ).drop_duplicates("identifier")
    df = df.merge(tcren, on="identifier", how="left")
    pan = pd.read_table(
        f"{MPREDPAN_DIR}/{ds}_mixtcrpred-pan_predicted.tsv",
        usecols=["identifier", "binder_prob"],
    ).drop_duplicates("identifier").rename(columns={"binder_prob": "binding_score_mixtcrpred-pan"})
    df = df.merge(pan, on="identifier", how="left")
    df["sample_in_train_mixtcrpred-pan"] = df["sample_in_train_mixtcrpred"]
    df["seen_in_mixtcrpred-pan"]         = df["seen_in_mixtcrpred"]
    assert df["identifier"].is_unique, f"{ds}: duplicate identifiers after load"
    return df

In [ ]:
# === Model lists + no-leakage helpers ===
ALL_MODELS    = ["t2pmhc-gcn", "t2pmhc-gat", "mixtcrpred", "mixtcrpred-pan",
                 "tulip-tcr", "tabr-bert", "ergo2", "panpep", "tcren"]
T2PMHC_MODELS = ["t2pmhc-gcn", "t2pmhc-gat"]
OTHER_MODELS  = ["mixtcrpred", "mixtcrpred-pan", "tulip-tcr", "tabr-bert",
                 "ergo2", "panpep", "tcren"]
# Models that cannot score unseen samples (no prediction -> NaN); rendered as NaN AUC.
UNSEEN_SKIP_MODELS = ["mixtcrpred"]


def _score(df, model):
    s = df[f"binding_score_{model}"]
    return -s if model in INVERTED_MODELS else s


def _no_leakage(df, train_models, score_models=None):
    if score_models is None:
        score_models = train_models
    mask = np.logical_and.reduce([df[f"sample_in_train_{m}"] == False for m in train_models])
    return df[mask].dropna(subset=[f"binding_score_{m}" for m in score_models])


def _split_by_seen(df, models, seen):
    flags = np.logical_and.reduce([df[f"seen_in_{m}"] == seen for m in models])
    return df[flags]


def _assert_clean(df, train_models, ds, label, score_models=None):
    if df.empty:
        return
    if score_models is None:
        score_models = train_models
    for m in train_models:
        leaked = int((df[f"sample_in_train_{m}"] == True).sum())
        assert leaked == 0, f"[{ds}/{label}] {leaked} leaked rows for {m}"
    for m in score_models:
        n_nan = int(df[f"binding_score_{m}"].isna().sum())
        assert n_nan == 0, f"[{ds}/{label}] {n_nan} NaN scores for {m}"

In [ ]:
# === tcrdock PAE helpers ===
# tcrdock_pae = raw pmhc_tcr_pae (lower = binder). No train-leakage/seen flags of its
# own, so the T2PMHC flags drive both the leakage filter and the seen/unseen split.
TCRDOCK_MODELS = ["tcrdock_pae"]
PALETTE_BASE_TCRDOCK = {**PALETTE_BASE, "tcrdock_pae": "#4d4d4d"}
# Lower = binder -> negate before AUC via _score. Idempotent on re-run.
INVERTED_MODELS = list(dict.fromkeys(list(INVERTED_MODELS) + TCRDOCK_MODELS))
TCRDOCK_MODELS_FULL = T2PMHC_MODELS + TCRDOCK_MODELS


def _attach_pae(df, ds):
    pae = pd.read_csv(f"{PAE_DIR}/{ds}_t2pmhc-gcn_predicted.tsv",
                      sep="\t", usecols=["identifier", "pmhc_tcr_pae"]).drop_duplicates("identifier")
    df = df.merge(pae, on="identifier", how="left")
    assert df["pmhc_tcr_pae"].notna().all(), f"{ds}: missing PAE after merge"
    df["binding_score_tcrdock_pae"] = df["pmhc_tcr_pae"]
    return df


def _tcrdock_subset(ds):
    df = pd.read_table(f"{ANALYZER_DIR_MAIN}/{ds}_analyzer_table.tsv").drop_duplicates("identifier")
    df = _attach_pae(df, ds)
    df = _no_leakage(df, T2PMHC_MODELS, T2PMHC_MODELS + ["tcrdock_pae"])
    ds_seen   = _split_by_seen(df, T2PMHC_MODELS, seen=True)
    ds_unseen = _split_by_seen(df, T2PMHC_MODELS, seen=False)
    _assert_clean(ds_seen,   T2PMHC_MODELS, ds, "seen/tcrdock",   T2PMHC_MODELS + ["tcrdock_pae"])
    _assert_clean(ds_unseen, T2PMHC_MODELS, ds, "unseen/tcrdock", T2PMHC_MODELS + ["tcrdock_pae"])
    return ds_seen, ds_unseen

In [ ]:
# === Pairwise no-leakage subsets incl. tcrdock ===
# For each baseline: subset filtered on {t2pmhc-gcn, t2pmhc-gat, baseline}, split
# Seen/Unseen. tcrdock uses the T2PMHC-driven split ("all" mode). Feeds panel (a).
TCRDOCK_OTHER_MODELS  = ["tcrdock_pae"]
PAIRWISE_OTHER_MODELS = OTHER_MODELS + TCRDOCK_OTHER_MODELS

pairwise_subsets = {}
for other in OTHER_MODELS:
    models = T2PMHC_MODELS + [other]
    unseen_skip         = [m for m in models if m in UNSEEN_SKIP_MODELS]
    unseen_score_models = [m for m in models if m not in unseen_skip]
    for ds in DATASETS:
        df = _load_analyzer(ds)
        df_seen_raw   = _no_leakage(df, models, models)
        df_unseen_raw = _no_leakage(df, models, unseen_score_models)
        ds_seen   = _split_by_seen(df_seen_raw,   models, seen=True)
        ds_unseen = _split_by_seen(df_unseen_raw, models, seen=False)
        _assert_clean(ds_seen,   models, ds, f"per-ds-seen/{other}",   models)
        _assert_clean(ds_unseen, models, ds, f"per-ds-unseen/{other}", unseen_score_models)
        pairwise_subsets[(ds, other, "Seen")]   = (ds_seen,   models,              ())
        pairwise_subsets[(ds, other, "Unseen")] = (ds_unseen, unseen_score_models, unseen_skip)

for ds in DATASETS:
    ds_seen, ds_unseen = _tcrdock_subset(ds)
    for other in TCRDOCK_OTHER_MODELS:
        models = T2PMHC_MODELS + [other]
        pairwise_subsets[(ds, other, "Seen")]   = (ds_seen,   models, ())
        pairwise_subsets[(ds, other, "Unseen")] = (ds_unseen, models, ())

In [ ]:
# === Panel (a): mean per-peptide AUC bars +/- 1 SD (horizontal) ===
# Each cell: AUC on the x-axis; Seen (top lane) and Unseen (bottom lane) stacked on y,
# separated by a thin divider line. Within a lane, three horizontal bars run from the axis
# floor (0.2) to the MEAN of the per-peptide AUCs -- t2pmhc-GCN (top), t2pmhc-GAT (middle),
# baseline (bottom). A simple thin dark whisker at each bar tip spans +/- 1 SD of that
# model's per-peptide AUC distribution (sample SD, ddof=1) -- the between-peptide spread,
# NOT a CI, so it does not shrink with peptide count. Peptides with a single binder class
# are dropped before averaging; N= is the number of valid peptides feeding the t2pmhc bars
# (drawn at each lane's outer edge, clear of the divider). The dashed line = 0.5 chance.
# Colours are shared with panel (b) via the figure-level legend.
# plot_mean_auc_strips(axes, transpose=True) swaps the grid: datasets on rows, models on
# columns (model names as horizontal column titles; dataset names as row labels).
BASELINE_DOT = "#808080"
DB_XLIM   = (0.2, 1.0)                       # shared AUC axis (x)
DB_XTICKS = np.array([0.2, 0.6, 1.0])        # coarse AUC ticks
_LANES = [(1, "Seen"), (0, "Unseen")]        # Seen on top, Unseen on bottom
_DY   = 0.28                                 # sub-row offset within a lane
_BARH = 0.24                                 # bar thickness (< _DY so adjacent bars keep a gap)

# Whisker (+/- 1 SD) styling. A plain thin dark-grey line -- simple, a little darker than
# the bars, and drawn on top so both ends stay readable without covering the bars.
_WHISKER_COLOR    = "#555555"
_WHISKER_LW       = 0.8
_WHISKER_CAPSIZE  = 1.5

# Canonical display names -- single source of truth reused by every panel/legend.
# "gcn"/"gat" (violin model ids) alias to the t2pmhc-GCN/GAT names.
DISPLAY_NAME = {
    "t2pmhc-gcn": "t2pmhc-GCN", "t2pmhc-gat": "t2pmhc-GAT",
    "gcn": "t2pmhc-GCN", "gat": "t2pmhc-GAT",
    "mixtcrpred": "MixTCRpred", "mixtcrpred-pan": "MixTCRpred-pan",
    "tulip-tcr": "TULIP-TCR", "tabr-bert": "TABR-BERT", "ergo2": "ERGO-II",
    "panpep": "PanPep", "tcren": "TCRen", "tcrdock_pae": "TCRdock-PAE",
    "baseline": "baseline",
}
# Full dataset names (no abbreviations); rendered small so column titles never collide.
DS_DISPLAY = {"public_set": "public", "immrep23": "immrep23",
              "epytope_viral": "epytope-viral", "iggytop_unseen": "iggytop-unseen"}


def _model_color(m, palette_base=None):
    palette_base = palette_base if palette_base is not None else PALETTE_BASE_TCRDOCK
    if m in ("t2pmhc-gcn", "t2pmhc-gat"):
        return palette_base[m.replace("t2pmhc-", "")]
    return palette_base.get(m, "#888888")


def shared_model_legend_handles():
    """Colour legend shared by panels (a) and (b): t2pmhc-GCN / t2pmhc-GAT / baseline."""
    return [
        Patch(facecolor=_model_color("t2pmhc-gcn"), edgecolor="none", label=DISPLAY_NAME["t2pmhc-gcn"]),
        Patch(facecolor=_model_color("t2pmhc-gat"), edgecolor="none", label=DISPLAY_NAME["t2pmhc-gat"]),
        Patch(facecolor=BASELINE_DOT,               edgecolor="none", label=DISPLAY_NAME["baseline"]),
    ]


def _pep_aucs(d, model):
    """Per-peptide AUCs for `model`; peptides with <2 binder classes are skipped."""
    aucs = []
    for _, g in d.groupby("peptide"):
        if g["binder"].nunique() < 2:
            continue
        aucs.append(roc_auc_score(g["binder"], _score(g, model)))
    return aucs


def _mean_sd(a):
    """(mean, sample SD ddof=1) of a list; SD is 0 for a single peptide."""
    a = np.asarray(a, dtype=float)
    return float(a.mean()), (float(a.std(ddof=1)) if len(a) > 1 else 0.0)


def _barh(ax, ms, y, color):
    """Horizontal bar from the axis floor to the mean, centred at `y`, with a +/- 1 SD
    whisker at the bar tip. `ms` is (mean, sd) or None (bar absent). The whisker is a plain
    thin dark-grey line (module-level _WHISKER_* knobs) -- simple, drawn on top of the bar."""
    if ms is None:
        return
    val, sd = ms
    ax.barh(y, width=val - DB_XLIM[0], left=DB_XLIM[0], height=_BARH,
            color=color, alpha=0.85, edgecolor="white", linewidth=0.8, zorder=3)
    if sd > 0:
        ax.errorbar([val], [y], xerr=[[sd], [sd]], fmt="none", ecolor=_WHISKER_COLOR,
                    elinewidth=_WHISKER_LW, capsize=_WHISKER_CAPSIZE,
                    capthick=_WHISKER_LW, zorder=4)


def _plot_meanstrip_cell(ax, mp_records, other, ds, show_ylabels, show_xticklabels,
                         col_title, ylabel_text, title_text,
                         ylabel_fontsize=9, title_fontsize=11, xtick_fontsize=8,
                         lanelabel_fontsize=8):
    # No shaded Unseen band -- it swallowed the grey baseline bars/whiskers. A single thin,
    # light divider at 0.5 separates the Seen (top) and Unseen (bottom) lanes on a white
    # ground; it is lighter than the SD whiskers so the two never read as the same element.
    ax.axhline(0.5, color="#cfcfcf", lw=1.0, zorder=0.5)      # Seen / Unseen lane divider
    for y, subset in _LANES:
        rec = mp_records[(ds, subset, other)]
        if rec is None:                       # empty/degenerate lane -> left blank
            continue
        _barh(ax, rec["gcn"], y + _DY, _model_color("t2pmhc-gcn"))   # top
        _barh(ax, rec["gat"], y,       _model_color("t2pmhc-gat"))   # middle
        _barh(ax, rec["oth"], y - _DY, BASELINE_DOT)                 # bottom
        if rec["oth"] is None:      # baseline cannot score this split (MixTCRpred/unseen)
            ax.text(DB_XLIM[0] + 0.25, y - _DY, "NA", fontsize=7, color="#555555",
                    ha="left", va="center", zorder=5)
        # N= at the lane's OUTER edge (Seen -> top, Unseen -> bottom) so it never sits on
        # the central 0.5 divider.
        n_y = 1.5 if subset == "Seen" else -0.5
        ax.text(DB_XLIM[0] + 0.02, n_y, f"N={rec['n']}", fontsize=6,
                color="#000000", ha="left", va="center", zorder=5)
    ax.set_xlim(*DB_XLIM)
    ax.set_xticks(DB_XTICKS)
    ax.tick_params(axis="x", labelsize=xtick_fontsize)
    ax.set_ylim(-0.65, 1.65)
    ax.set_yticks([0, 1])
    ax.axvline(0.5, color="grey", linestyle="--", lw=0.8, alpha=0.6, zorder=1)
    ax.set_axisbelow(True)
    if show_ylabels:
        # Seen/Unseen lane labels on the leftmost column; model name sits further out.
        # NOTE: with sharey=True the y-axis formatter is shared across the whole grid, so
        # setting real tick labels here and only *hiding* them elsewhere (tick_params below)
        # is what keeps Unseen/Seen visible -- calling set_yticklabels([]) on other columns
        # would wipe the shared formatter and blank these labels everywhere.
        ax.set_yticklabels(["Unseen", "Seen"], fontsize=lanelabel_fontsize)
        ax.set_ylabel(ylabel_text, rotation=0, ha="right", va="center",
                      fontsize=ylabel_fontsize, labelpad=26)
    else:
        ax.tick_params(axis="y", labelleft=False)
    # Same shared-axis trap on x: the bottom row shows the AUC values via the default
    # formatter, so other rows must only HIDE their x labels (tick_params), never call
    # set_xticklabels([]) which would blank the shared formatter for the bottom row too.
    if not show_xticklabels:
        ax.tick_params(axis="x", labelbottom=False)
    if col_title:
        ax.set_title(title_text, fontsize=title_fontsize)


def plot_mean_auc_strips(axes, transpose=False, others=None, show_col_titles=True):
    """Draw the mean per-peptide AUC bar grid into a provided axes array.

    Default grid: rows = baselines (PAIRWISE_OTHER_MODELS), cols = datasets (dataset
    titles on top, model names as row labels). transpose=True: rows = datasets,
    cols = models (model names as horizontal column titles; dataset names as row labels).
    Each cell: horizontal mean per-peptide AUC bars +/- 1 SD, Seen/Unseen lanes on y.
    """
    others = PAIRWISE_OTHER_MODELS if others is None else others
    mp_records = {}   # (ds, subset, other) -> {"gcn":(m,sd),"gat":(m,sd),"oth":(m,sd)|None,"n":..} | None
    for subset in ["Seen", "Unseen"]:
        for ds in DATASETS:
            for other in others:
                d, score_models, nan_models = pairwise_subsets[(ds, other, subset)]
                if d.empty:
                    mp_records[(ds, subset, other)] = None
                    continue
                gcn = _pep_aucs(d, "t2pmhc-gcn")
                gat = _pep_aucs(d, "t2pmhc-gat")
                if not gcn:                    # no scorable peptide -> lane absent
                    mp_records[(ds, subset, other)] = None
                    continue
                oth = None if other in nan_models else _pep_aucs(d, other)
                mp_records[(ds, subset, other)] = {
                    "gcn": _mean_sd(gcn), "gat": _mean_sd(gat),
                    "oth": (_mean_sd(oth) if oth else None), "n": len(gcn)}
    row_items = DATASETS if transpose else others
    col_items = others if transpose else DATASETS
    n_rows = len(row_items)
    title_fs  = 8 if transpose else 11        # 8 horizontal model columns need a small title
    ylabel_fs = 9                             # model names kept small so the cells get the space
    xtick_fs  = 7 if transpose else 8
    for r, row_key in enumerate(row_items):
        for c, col_key in enumerate(col_items):
            other = col_key if transpose else row_key
            ds    = row_key if transpose else col_key
            title_text  = DISPLAY_NAME.get(other, other) if transpose else DS_DISPLAY.get(ds, ds)
            ylabel_text = DS_DISPLAY.get(ds, ds) if transpose else DISPLAY_NAME.get(other, other)
            _plot_meanstrip_cell(
                axes[r, c], mp_records, other, ds,
                show_ylabels=(c == 0), show_xticklabels=(r == n_rows - 1), col_title=(r == 0 and show_col_titles),
                ylabel_text=ylabel_text, title_text=title_text,
                ylabel_fontsize=ylabel_fs, title_fontsize=title_fs, xtick_fontsize=xtick_fs)

In [ ]:
# ============================================================================
# Figure 5 (t2pmhc variants only) -- t2pmhc-GCN vs t2pmhc-GAT.
# Leakage removed for the T2PMHC models ONLY (filter on T2PMHC_MODELS, no baseline),
# so this is the MAXIMAL shared, leakage-free subset for the two variants -- wider than
# the pairwise subsets in the figure above, which additionally drop rows leaked into each
# baseline. Same mean per-peptide AUC bar conventions (+/- 1 SD whisker; Seen top / Unseen
# bottom lanes); 1 row x 4 datasets, two bars per lane. Reuses Fig 1's _barh / _pep_aucs /
# _mean_sd / _model_color and the shared lane/axis constants.
# ============================================================================
t2pmhc_only_subsets = {}
for ds in DATASETS:
    df = _load_analyzer(ds)
    clean = _no_leakage(df, T2PMHC_MODELS, T2PMHC_MODELS)   # only t2pmhc leakage removed
    ds_seen   = _split_by_seen(clean, T2PMHC_MODELS, seen=True)
    ds_unseen = _split_by_seen(clean, T2PMHC_MODELS, seen=False)
    _assert_clean(ds_seen,   T2PMHC_MODELS, ds, "t2pmhc-only/Seen")
    _assert_clean(ds_unseen, T2PMHC_MODELS, ds, "t2pmhc-only/Unseen")
    t2pmhc_only_subsets[(ds, "Seen")]   = ds_seen
    t2pmhc_only_subsets[(ds, "Unseen")] = ds_unseen


def _plot_t2pmhc_only_cell(ax, ds, show_ylabels, title_text, show_xticklabels=True, row_label=None):
    """One dataset cell: t2pmhc-GCN (upper) / t2pmhc-GAT (lower) mean per-peptide AUC bars,
    Seen (top lane) / Unseen (bottom lane). Two bars per lane -- no baseline row."""
    ax.axhline(0.5, color="#cfcfcf", lw=1.0, zorder=0.5)          # Seen / Unseen divider
    for y, subset in _LANES:
        d = t2pmhc_only_subsets[(ds, subset)]
        gcn = _pep_aucs(d, "t2pmhc-gcn")
        gat = _pep_aucs(d, "t2pmhc-gat")
        if not gcn:                                              # no scorable peptide -> blank
            continue
        _barh(ax, _mean_sd(gcn), y + _DY / 2, _model_color("t2pmhc-gcn"))   # upper bar
        _barh(ax, _mean_sd(gat), y - _DY / 2, _model_color("t2pmhc-gat"))   # lower bar
        n_y = 1.5 if subset == "Seen" else -0.5                  # N= at lane's outer edge
        ax.text(DB_XLIM[0] + 0.02, n_y, f"N={len(gcn)}", fontsize=6,
                color="#000000", ha="left", va="center", zorder=5)
    # ax.set_xlim(0.5, 1.0)
    # ax.set_xticks(np.arange(0.5,1.0,0.2))
    ax.set_xlim(*DB_XLIM)
    ax.set_xticks(DB_XTICKS)
    ax.tick_params(axis="x", labelsize=8)
    if not show_xticklabels:                 # shared-axis grid: only bottom row labels x
        ax.tick_params(axis="x", labelbottom=False)
    ax.set_ylim(-0.65, 1.65)
    ax.set_yticks([0, 1])
    ax.axvline(0.5, color="grey", linestyle="--", lw=0.8, alpha=0.6, zorder=1)
    ax.set_axisbelow(True)
    # sharey=True: set real labels on col 0 and only HIDE them elsewhere (see Fig 1 note).
    if show_ylabels:
        ax.set_yticklabels(["Unseen", "Seen"], fontsize=8)
        if row_label:                        # row name on the left, like the baseline rows in (b)
            ax.set_ylabel(row_label, rotation=0, ha="right", va="center",
                          fontsize=9, labelpad=26)
    else:
        ax.tick_params(axis="y", labelleft=False)
    ax.set_title(title_text, fontsize=11)

In [ ]:
# ============================================================================
# Figure 5 (combined) -- single stacked figure
#   (a) t2pmhc variants only: t2pmhc-GCN vs t2pmhc-GAT mean per-peptide AUC bars,
#       leakage removed for the t2pmhc models ONLY (maximal shared subset). This is the
#       TOP ROW of the same shared-axis AUC grid as (b) -- same 4 dataset columns, same
#       AUC x-axis -- set off by a slightly larger gap so it reads as one figure, not two.
#   (b) mean per-peptide AUC bars +/- 1 SD (8 baselines x 4 datasets).
#   (c) combined violin plot: calibrated binding probability, stCRDab, Seen/Unseen.
# Panel (a) reuses _plot_t2pmhc_only_cell / t2pmhc_only_subsets from the cell above.
# ============================================================================
figC = plt.figure(figsize=figsize("double", 232), constrained_layout=True)
sf_grid, sf_c = figC.subfigures(2, 1, height_ratios=[9.25, 2.6])

n_base = len(PAIRWISE_OTHER_MODELS)
n_ds   = len(DATASETS)

# One shared-axis AUC grid: row 0 = t2pmhc-only (a); an empty spacer row gives a bigger gap
# than the inter-baseline gaps; then the 8 baseline rows (b). All cells share x AND y, so
# the dataset columns and the AUC axis line up across a and b -- a is literally the grid's
# top row, not a second, differently-scaled plot. Dataset titles + x-tick labels appear
# once (titles atop a; ticks under the bottom baseline row).
# >>> a-b spacing knob <<< AB_GAP = weight of the empty spacer row between panels (a) and
# (b). Each bar row = 1.0, so AB_GAP=1.0 is one full row of gap; AB_GAP=0.0 leaves only the
# normal inter-row gap (a sits as close to b as the baseline rows are to each other). That
# normal gap is a FLOOR set by constrained_layout's row pad -- to pull a and b tighter than
# the baseline rows, also lower the pad via the commented line below. Edit + re-run to taste.
AB_GAP = 0.25
gs = sf_grid.add_gridspec(n_base + 2, n_ds, height_ratios=[1.0, AB_GAP] + [1.0] * n_base)
# figC.set_constrained_layout_pads(hspace=0.0, h_pad=0.01)  # uncomment to tighten ALL row gaps
_ref = None
def _cell(r, c):
    global _ref
    ax = sf_grid.add_subplot(gs[r, c], sharex=_ref, sharey=_ref)
    if _ref is None:
        _ref = ax
    return ax

# --- (a) t2pmhc-only top row ---
axes_t = [_cell(0, c) for c in range(n_ds)]
for c, ds in enumerate(DATASETS):
    _plot_t2pmhc_only_cell(axes_t[c], ds, show_ylabels=(c == 0),
                           title_text=DS_DISPLAY.get(ds, ds), show_xticklabels=False,
                           row_label="t2pmhc-intern")

# --- (b) baseline grid (rows offset by the spacer row at index 1) ---
axes_a = np.empty((n_base, n_ds), dtype=object)
for r in range(n_base):
    for c in range(n_ds):
        axes_a[r, c] = _cell(r + 2, c)
plot_mean_auc_strips(axes_a, show_col_titles=False)   # dataset titles already on the a row
sf_grid.supxlabel("Mean per-peptide AUC", fontsize=10)

# --- (c) combined violins ---
axes_c = sf_c.subplots(1, 2, sharey=True)
plot_violins_combined(combined_seen_df,   ax=axes_c[0])
plot_violins_combined(combined_unseen_df, ax=axes_c[1])
axes_c[0].set_title("Seen")
axes_c[1].set_title("Unseen")
# Cap at 1.0: binding probability is bounded, so clip any calibrated values that spill
# above 1.0 -- boxes/violins never flow past the top (probability ticks stay 0..1.0).
for ax in axes_c:
    ax.set_ylim(0, 1.0)
    ax.set_xticklabels([DISPLAY_NAME.get(t.get_text(), t.get_text())
                        for t in ax.get_xticklabels()], fontsize=9)
axes_c[1].yaxis.set_visible(False)
axes_c[1].spines['left'].set_visible(False)
sf_c.supxlabel("Model", fontsize=10)
sf_c.supylabel("Binding probability", fontsize=10)

# Shared colour legend (all panels) at the very top of the figure. Kept small/tight so the
# space goes to the plots rather than the key.
figC.legend(handles=shared_model_legend_handles(), loc="outside upper center",
            ncol=3, frameon=False, fontsize=9, handlelength=1.1, handleheight=1.0,
            handletextpad=0.5, columnspacing=1.4, borderaxespad=0.1)

# Panel labels a / b / c. Placed in the far-LEFT figure margin (x) at the TOP of each block
# (y) via a blended transform: figure-fraction x + that block's axes-fraction y. va="top"
# makes each letter hang DOWN into its own panel's left margin, so it never overhangs the
# a-b gap -- an overhanging label there makes constrained_layout inflate the gap and blocks
# AB_GAP from closing. This tracks the layout automatically as AB_GAP changes.
from matplotlib.transforms import blended_transform_factory
PANEL_LABEL_X = 0.006            # figure-fraction x for all panel letters; nudge to taste
def _panel_label(letter, ax):
    trans = blended_transform_factory(figC.transFigure, ax.transAxes)
    figC.text(PANEL_LABEL_X, 1.0, letter, transform=trans,
              fontsize=16, va="top", ha="left")
_panel_label("a", axes_t[0])
_panel_label("b", axes_a[0, 0])
_panel_label("c", axes_c[0])

save_figure(figC, "figure5_benchmark", subdir="fig5")
plt.show()

In [ ]:
# ============================================================================
# Per-dataset mean per-peptide AUC delta: each t2pmhc variant minus each competitor.
# For every comparison (dataset x competitor x Seen/Unseen lane x variant), delta =
# mean(variant per-peptide AUCs) - mean(competitor per-peptide AUCs), computed on the
# SAME shared, leakage-free pairwise subset `d` (both means run over the same peptides,
# since _pep_aucs skips single-binder-class peptides identically -> N is shared). A
# positive delta means the t2pmhc variant beats the competitor on that dataset/lane.
# Seen and Unseen are kept as separate rows. Reuses pairwise_subsets / _pep_aucs /
# _mean_sd / DS_DISPLAY / DISPLAY_NAME -- no new plotting, just the numeric table.
# ============================================================================
rows = []
for ds in DATASETS:
    for other in PAIRWISE_OTHER_MODELS:
        for subset in ["Seen", "Unseen"]:
            d, score_models, nan_models = pairwise_subsets[(ds, other, subset)]
            comp_aucs = None if (other in nan_models or d.empty) else _pep_aucs(d, other)
            for variant in T2PMHC_MODELS:
                var_aucs = None if d.empty else _pep_aucs(d, variant)
                if not var_aucs or not comp_aucs:        # unscorable lane (e.g. mixtcrpred/Unseen)
                    v_mean = comp_mean = delta = np.nan
                    n = 0
                else:
                    v_mean,    _ = _mean_sd(var_aucs)
                    comp_mean, _ = _mean_sd(comp_aucs)
                    delta = v_mean - comp_mean
                    n = len(var_aucs)                    # == len(comp_aucs) on the shared subset
                rows.append({
                    "dataset":        DS_DISPLAY.get(ds, ds),
                    "lane":           subset,
                    "variant":        DISPLAY_NAME.get(variant, variant),
                    "competitor":     DISPLAY_NAME.get(other, other),
                    "n_peptides":     n,
                    "auc_variant":    v_mean,
                    "auc_competitor": comp_mean,
                    "delta":          delta,
                })

delta_df = pd.DataFrame(rows, columns=[
    "dataset", "lane", "variant", "competitor",
    "n_peptides", "auc_variant", "auc_competitor", "delta"])
display(delta_df.round(3))

delta_df.to_csv(RESULTS / "delta_auc_table.tsv", sep="\t", index=False)

In [ ]:
# ============================================================================
# Panel (a) as a table -- mean per-peptide AUC +/- 1 SD for every bar.
# One row per (baseline, dataset, Seen/Unseen); columns give t2pmhc-GCN / t2pmhc-GAT /
# baseline as "mean ± sd" over the SAME leakage-free, seen/unseen-split per-peptide AUCs
# that feed the bars (pairwise_subsets + _pep_aucs / _mean_sd). A blank baseline cell means
# that model cannot score that split (e.g. MixTCRpred on unseen) -- the omitted NaN bar.
# ============================================================================
def _fmt_meansd(a):
    """'mean ± sd' from a list of per-peptide AUCs, or '' if the bar is absent."""
    if not a:
        return ""
    m, sd = _mean_sd(a)
    return f"{m:.3f} ± {sd:.3f}"


_panelA_rows = []
for other in PAIRWISE_OTHER_MODELS:
    for ds in DATASETS:
        for subset in ["Seen", "Unseen"]:
            d, score_models, nan_models = pairwise_subsets[(ds, other, subset)]
            if d.empty:
                continue
            gcn = _pep_aucs(d, "t2pmhc-gcn")
            if not gcn:                          # no scorable peptide -> no bar group
                continue
            gat = _pep_aucs(d, "t2pmhc-gat")
            oth = None if other in nan_models else _pep_aucs(d, other)
            _panelA_rows.append({
                "baseline":     DISPLAY_NAME.get(other, other),
                "dataset":      DS_DISPLAY.get(ds, ds),
                "split":        subset,
                "N_peptides":   len(gcn),
                "t2pmhc-GCN":   _fmt_meansd(gcn),
                "t2pmhc-GAT":   _fmt_meansd(gat),
                "baseline_AUC": _fmt_meansd(oth) if oth else "",
            })

panelA_table = pd.DataFrame(_panelA_rows)

# Sanity checks: one row per plotted (baseline, dataset, split) group, positive N, and every
# non-empty cell parses as a mean in [0, 1].
assert not panelA_table.duplicated(["baseline", "dataset", "split"]).any(), "duplicate cell"
assert (panelA_table["N_peptides"] > 0).all(), "empty group leaked into the table"
_means = pd.to_numeric(
    panelA_table[["t2pmhc-GCN", "t2pmhc-GAT", "baseline_AUC"]]
    .replace("", np.nan).stack().str.split("±").str[0], errors="raise")

_panelA_csv = str(RESULTS / "figure5_panelA_auc_table.csv")
panelA_table.to_csv(_panelA_csv, index=False)
print(f"panel (a) table: {len(panelA_table)} rows -> {_panelA_csv}")
panelA_table

## Significance testing

Paired two-sided Wilcoxon signed-rank tests (t2pmhc-GCN / t2pmhc-GAT vs each competitor on shared, leakage-free per-peptide AUCs), Holm-corrected within families. Reuses the `pairwise_subsets` / `_score` / `DISPLAY_NAME` built above. STEP 2 asserts the per-peptide means reproduce the two tables saved above.

In [ ]:
# --- Per-peptide AUC helper ------------------------------------------------------------
def _peptide_aucs(df, score_models):
    """Tidy per-peptide ROC AUC for every model in `score_models`.

    Returns columns ["Model", "peptide", "AUC"]. A peptide with a single binder class has
    an undefined AUC and is emitted as NaN for every model (so it drops out identically).
    """
    rows = []
    if df.empty:
        return pd.DataFrame(rows, columns=["Model", "peptide", "AUC"])
    for pep, g in df.groupby("peptide"):
        single_class = g["binder"].nunique() < 2
        for m in score_models:
            auc = np.nan if single_class else roc_auc_score(g["binder"], _score(g, m))
            rows.append({"Model": m, "peptide": pep, "AUC": auc})
    return pd.DataFrame(rows, columns=["Model", "peptide", "AUC"])

In [ ]:
# delta_auc_table.tsv + figure5_panelA_auc_table.csv were written to ../results/ by the figure
# cells above; the STEP-2 gate reads them back from there, and results are written there too.
BF_DIR      = str(RESULTS)
RESULTS_DIR = str(RESULTS)

# === Significance tests — Cell A: module-level per-peptide AUC frame ===
# Generalises panel_b_df (cell 13) from (iggytop_unseen, Unseen) to EVERY
# (dataset, split, competitor) cell, using the identical pairwise_subsets (cell 11)
# + _peptide_aucs (cell 10) construction the plots use. No AUC is recomputed in any
# new way; every value is _peptide_aucs on the same leakage-free subset. For a given
# (ds, other, split) the subset df is identical for t2pmhc-gcn, t2pmhc-gat AND the
# competitor, so their per-peptide AUCs share one peptide index -> the pairing is an
# inner join on `peptide`, and peptides with <2 binder classes drop identically.
_ppa_rows = []
_ppa_lane_log = []   # absent/empty lanes (dataset, split, competitor, reason)

for ds in DATASETS:
    for other in PAIRWISE_OTHER_MODELS:
        for split in ["Seen", "Unseen"]:
            d, score_models, nan_models = pairwise_subsets[(ds, other, split)]
            comp = DISPLAY_NAME.get(other, other)
            ds_disp = DS_DISPLAY.get(ds, ds)
            if d.empty:
                _ppa_lane_log.append((ds_disp, split, comp, "empty leakage-free subset (no rows)"))
                continue
            comp_absent = other in nan_models            # competitor cannot score this split
            if comp_absent:
                _ppa_lane_log.append((ds_disp, split, comp,
                                      "competitor cannot score this split (UNSEEN_SKIP)"))
            # Score only models that actually predicted on this subset. When the competitor
            # cannot score the split (comp_absent), its binding_score column is NaN here, so
            # feeding it to roc_auc_score would raise "Input contains NaN" — drop it. The two
            # t2pmhc models are always scorable; peptides with <2 binder classes stay NaN.
            score_list = T2PMHC_MODELS + ([] if comp_absent else [other])
            pa = _peptide_aucs(d, score_list)             # tidy ["Model","peptide","AUC"]
            for _, r in pa.iterrows():
                m = r["Model"]
                _ppa_rows.append({
                    "dataset":        ds_disp,
                    "dataset_raw":    ds,
                    "split":          split,
                    "competitor":     comp,
                    "competitor_raw": other,
                    "model":          DISPLAY_NAME.get(m, m),
                    "model_raw":      m,
                    "peptide":        r["peptide"],
                    "AUC":            r["AUC"],
                })

per_peptide_auc_df = pd.DataFrame(_ppa_rows)
per_peptide_auc_df["scorable"] = per_peptide_auc_df["AUC"].notna()

# Guardrail: within each (dataset, split, competitor) the scorable peptide set must be
# identical across models — this is what makes the paired inner-join well defined.
for (dd, sp, cc), g in per_peptide_auc_df[per_peptide_auc_df["scorable"]].groupby(
        ["dataset", "split", "competitor"]):
    sets = g.groupby("model")["peptide"].agg(lambda s: frozenset(s))
    assert sets.nunique() == 1, f"peptide sets differ across models in {(dd, sp, cc)}"

# Log every dropped (unscorable) peptide with reason — no silent drops.
ppa_drops = (per_peptide_auc_df.loc[~per_peptide_auc_df["scorable"],
                                    ["dataset", "split", "competitor", "peptide"]]
             .drop_duplicates()
             .assign(reason="<2 binder classes in subset"))
ppa_lanes = pd.DataFrame(_ppa_lane_log,
                         columns=["dataset", "split", "competitor", "reason"]).drop_duplicates()

print(f"per_peptide_auc_df: {len(per_peptide_auc_df)} rows | "
      f"{int(per_peptide_auc_df['scorable'].sum())} scorable | "
      f"{int((~per_peptide_auc_df['scorable']).sum())} unscorable peptide-rows")
print(f"dropped (unscorable) peptides: {len(ppa_drops)} | absent/empty lanes: {len(ppa_lanes)}")
print("\n-- dropped peptides (excluded from tests) --")
print(ppa_drops.to_string(index=False) if len(ppa_drops) else "(none)")
print("\n-- absent / empty lanes (no test possible) --")
print(ppa_lanes.to_string(index=False) if len(ppa_lanes) else "(none)")
per_peptide_auc_df.tail(12)

In [ ]:
# === Cell B: STEP 2 sanity gate — reproduce the figure's mean per-peptide AUCs ===
# The tests must run on the SAME numbers the figure reports. Confirm the per-subset means
# of per_peptide_auc_df exactly reproduce the two saved mean tables:
#   delta_auc_table.tsv        (cell 16, full precision  -> strict tol 1e-9)
#   figure5_panelA_auc_table.csv (cell 22, rounded 3 dp  -> tol 1e-3)
# Also confirm per-subset scorable N matches each table's peptide count. Gate: assert.
_means = (per_peptide_auc_df[per_peptide_auc_df["scorable"]]
          .groupby(["dataset", "split", "competitor", "model"])
          .agg(mean_auc=("AUC", "mean"), N=("peptide", "nunique"))
          .reset_index())

def _lookup_mean(dataset, split, competitor, model_display):
    sel = _means[(_means.dataset == dataset) & (_means.split == split) &
                 (_means.competitor == competitor) & (_means.model == model_display)]
    return (float(sel["mean_auc"].iloc[0]), int(sel["N"].iloc[0])) if len(sel) else (np.nan, 0)

# --- vs delta_auc_table.tsv (competitor-scorable lanes only; full precision) ---
_delta = pd.read_csv(f"{BF_DIR}/delta_auc_table.tsv", sep="\t")
_rows = []
for _, r in _delta[_delta["n_peptides"] > 0].iterrows():
    fm_v, n_v = _lookup_mean(r["dataset"], r["lane"], r["competitor"], r["variant"])
    fm_c, n_c = _lookup_mean(r["dataset"], r["lane"], r["competitor"], r["competitor"])
    _rows.append({"dataset": r["dataset"], "split": r["lane"], "competitor": r["competitor"],
                  "variant": r["variant"], "N_table": int(r["n_peptides"]), "N_frame": n_v,
                  "auc_variant_table": r["auc_variant"], "auc_variant_frame": fm_v,
                  "d_variant": abs(r["auc_variant"] - fm_v),
                  "auc_comp_table": r["auc_competitor"], "auc_comp_frame": fm_c,
                  "d_comp": abs(r["auc_competitor"] - fm_c)})
delta_recon = pd.DataFrame(_rows)
_bad_d = delta_recon[(delta_recon["d_variant"] > 1e-9) |
                     (delta_recon["d_comp"].fillna(0) > 1e-9) |
                     (delta_recon["N_table"] != delta_recon["N_frame"])]
print(f"delta_auc_table reconstruction: {len(delta_recon)} checked, {len(_bad_d)} mismatches (tol 1e-9)")
if len(_bad_d):
    display(_bad_d)
assert len(_bad_d) == 0, "STEP 2 gate FAILED vs delta_auc_table.tsv"

# --- vs figure5_panelA_auc_table.csv (rounded 3 dp) ---
_panelA = pd.read_csv(f"{BF_DIR}/figure5_panelA_auc_table.csv")
def _parse_meansd(s):
    if pd.isna(s) or str(s).strip() == "":
        return np.nan
    return float(str(s).split("±")[0])
_rows = []
for _, r in _panelA.iterrows():
    for col, disp in [("t2pmhc-GCN", "t2pmhc-GCN"), ("t2pmhc-GAT", "t2pmhc-GAT"),
                      ("baseline_AUC", r["baseline"])]:
        tab = _parse_meansd(r[col])
        if np.isnan(tab):
            continue
        fm, n_fr = _lookup_mean(r["dataset"], r["split"], r["baseline"], disp)
        _rows.append({"dataset": r["dataset"], "split": r["split"], "competitor": r["baseline"],
                      "model": disp, "N_table": int(r["N_peptides"]), "N_frame": n_fr,
                      "auc_table": tab, "auc_frame": fm, "absdiff": abs(tab - fm)})
panelA_recon = pd.DataFrame(_rows)
_bad_a = panelA_recon[(panelA_recon["absdiff"] > 1e-3) |
                      (panelA_recon["N_table"] != panelA_recon["N_frame"])]
print(f"figure5_panelA reconstruction: {len(panelA_recon)} checked, {len(_bad_a)} mismatches (tol 1e-3)")
if len(_bad_a):
    display(_bad_a)
assert len(_bad_a) == 0, "STEP 2 gate FAILED vs figure5_panelA_auc_table.csv"

print("\nSTEP 2 PASSED: per-peptide means reproduce BOTH figure tables and N matches. "
      "Side-by-side (delta table) sample:")
display(delta_recon[["dataset", "split", "competitor", "variant", "N_table", "N_frame",
                     "auc_variant_table", "auc_variant_frame",
                     "auc_comp_table", "auc_comp_frame"]].round(6).head(10))

In [ ]:
# === Cell C: test helpers (all seeds fixed) ===
import inspect
from scipy.stats import wilcoxon, rankdata

_BOOT_N    = 10000
_BOOT_SEED = 0
# scipy renamed the exact/approx selector `mode`->`method` around 1.7; support both.
_WILCOX_KW = "method" if "method" in inspect.signature(wilcoxon).parameters else "mode"


def _signed_rank_report(diffs, alternative="two-sided"):
    """Wilcoxon signed-rank on `diffs` (model - reference), plus matched-pairs
    rank-biserial. Zeros dropped ('wilcox' zero_method); exact test when the number of
    nonzero pairs < 15, else normal approximation. Positive rank-biserial => model higher."""
    diffs = np.asarray(diffs, dtype=float)
    n_pairs = len(diffs)
    n_zero  = int((diffs == 0).sum())
    nz      = diffs[diffs != 0]
    n_eff   = len(nz)
    if n_eff == 0:
        return dict(n_pairs=n_pairs, n_zero=n_zero, n_effective=0, zero_method="wilcox",
                    test_method=None, statistic=np.nan, p=np.nan, rank_biserial=np.nan)
    ranks   = rankdata(np.abs(nz))
    w_plus  = float(ranks[nz > 0].sum())
    w_minus = float(ranks[nz < 0].sum())
    rbc     = (w_plus - w_minus) / (w_plus + w_minus)
    method  = "exact" if n_eff < 15 else "approx"
    stat, p = wilcoxon(nz, alternative=alternative, zero_method="wilcox",
                       correction=False, **{_WILCOX_KW: method})
    return dict(n_pairs=n_pairs, n_zero=n_zero, n_effective=n_eff, zero_method="wilcox",
                test_method=method, statistic=float(stat), p=float(p), rank_biserial=rbc)


def _median_diff_ci(diffs, n_boot=_BOOT_N, seed=_BOOT_SEED):
    """Median paired difference + 95% percentile bootstrap CI (seeded; resamples pairs
    with replacement, includes zero differences)."""
    diffs = np.asarray(diffs, dtype=float)
    if len(diffs) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(diffs), size=(n_boot, len(diffs)))
    boot_med = np.median(diffs[idx], axis=1)
    return (float(np.median(diffs)),
            float(np.percentile(boot_med, 2.5)),
            float(np.percentile(boot_med, 97.5)))


def _paired_pep(model_disp, comp_disp, dataset, split):
    """Explicit inner join on peptide id of the two models' per-peptide AUCs WITHIN the
    competitor's leakage-free subset — never across subsets. Returns (peptides, x, y)."""
    sub = per_peptide_auc_df[(per_peptide_auc_df.dataset == dataset) &
                             (per_peptide_auc_df.split == split) &
                             (per_peptide_auc_df.competitor == comp_disp) &
                             (per_peptide_auc_df.scorable)]
    a = sub[sub.model == model_disp][["peptide", "AUC"]].rename(columns={"AUC": "x"})
    b = sub[sub.model == comp_disp][["peptide", "AUC"]].rename(columns={"AUC": "y"})
    j = a.merge(b, on="peptide", how="inner")
    return j["peptide"].to_numpy(), j["x"].to_numpy(), j["y"].to_numpy(), (len(a), len(b))


print(f"helpers ready | bootstrap n={_BOOT_N} seed={_BOOT_SEED} | scipy selector='{_WILCOX_KW}'")

In [ ]:
# === Cell D: STEP 3 — run the paired tests on the plotting data ===
# For each (dataset, split, competitor): paired two-sided Wilcoxon of t2pmhc-GCN and
# t2pmhc-GAT vs the competitor on the SHARED per-peptide pairs (inner join on peptide
# within that competitor's leakage-free subset). Every skipped lane is logged with a reason.
_results  = []
_skip_log = []

for ds_raw in DATASETS:
    ds = DS_DISPLAY.get(ds_raw, ds_raw)
    for other in PAIRWISE_OTHER_MODELS:
        comp = DISPLAY_NAME.get(other, other)
        for split in ["Seen", "Unseen"]:
            # ---------- paired: GCN / GAT vs competitor ----------
            for model_raw in T2PMHC_MODELS:
                model = DISPLAY_NAME.get(model_raw, model_raw)
                peps, x, y, (na, nb) = _paired_pep(model, comp, ds, split)
                n = len(peps)
                if n < 1:
                    _skip_log.append((ds, split, comp, model, "paired",
                                      "no shared scorable pairs (competitor absent/empty)"))
                    continue
                if not (na == nb == n):
                    print(f"  WARN join mismatch {ds}/{split} {model} vs {comp}: "
                          f"model={na} comp={nb} joined={n}")
                diffs = x - y
                rep = _signed_rank_report(diffs, "two-sided")
                med, lo, hi = _median_diff_ci(diffs)
                print(f"[paired] {ds}/{split}  {model} vs {comp}: joined N = {n}")
                _results.append(dict(
                    test_kind="paired_vs_competitor",
                    family=f"{model} vs competitors | {ds} | {split}",
                    model=model, competitor=comp, dataset=ds, split=split, N=n,
                    n_zero_diff=rep["n_zero"], n_effective=rep["n_effective"],
                    median_model=float(np.median(x)), median_reference=float(np.median(y)),
                    median_diff=med, ci_low=lo, ci_high=hi,
                    statistic=rep["statistic"], p_raw=rep["p"],
                    rank_biserial=rep["rank_biserial"],
                    zero_method=rep["zero_method"], test_method=rep["test_method"]))

results_df = pd.DataFrame(_results)
skips_df   = pd.DataFrame(_skip_log,
                          columns=["dataset", "split", "competitor", "model", "test", "reason"])
print(f"\nran {len(results_df)} paired tests | {len(skips_df)} lanes skipped")
print("\n-- skipped lanes (logged, no silent drops) --")
print(skips_df.to_string(index=False) if len(skips_df) else "(none)")

In [ ]:
# === Cell E: STEP 4 — Holm correction within explicit families ===
# Families (each corrected independently, holding the family-wise error rate):
#   * GCN-vs-competitor comparisons inside one (dataset, split)      -> "t2pmhc-GCN vs competitors | ds | split"
#   * GAT-vs-competitor comparisons inside one (dataset, split)      -> "t2pmhc-GAT vs competitors | ds | split"
# The `family` column (set in cell D) names each family verbatim.
def _holm(pvals):
    """Holm-Bonferroni step-down adjusted p-values (monotone, capped at 1)."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for i, idx in enumerate(order):
        running = max(running, (m - i) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj

results_df["p_holm"] = np.nan
for fam in results_df["family"].unique():
    sub = results_df[(results_df["family"] == fam) & results_df["p_raw"].notna()]
    if sub.empty:
        continue
    results_df.loc[sub.index, "p_holm"] = _holm(sub["p_raw"].to_numpy())

print("Holm adjustment applied within", results_df["family"].nunique(), "families.")
print("raw vs Holm-adjusted p (grouped by family):")
for fam, g in results_df.sort_values(["family", "p_raw"]).groupby("family"):
    print(f"\nFAMILY: {fam}  (m={g['p_raw'].notna().sum()})")
    print(g[["competitor", "model", "N", "p_raw", "p_holm"]]
          .to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
# === Cell F: STEP 5 — tidy output, summary, Methods paragraph ===
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

results_df["underpowered"] = results_df["N"] < 10          # flag small-N comparisons
_col_order = ["test_kind", "family", "dataset", "split", "model", "competitor",
              "N", "n_effective", "n_zero_diff", "underpowered",
              "median_model", "median_reference", "median_diff", "ci_low", "ci_high",
              "statistic", "p_raw", "p_holm", "rank_biserial", "zero_method", "test_method"]
results_df = results_df[_col_order].sort_values(
    ["test_kind", "dataset", "competitor", "split", "model"]).reset_index(drop=True)

_out = f"{RESULTS_DIR}/significance_tests.csv"
results_df.to_csv(_out, index=False)
print(f"wrote {len(results_df)} rows -> {os.path.abspath(_out)}")

_show = ["dataset", "split", "model", "competitor", "N", "underpowered",
         "median_model", "median_reference", "median_diff", "ci_low", "ci_high",
         "p_raw", "p_holm", "rank_biserial"]

def _print_block(df, title):
    print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)
    print(df[_show].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

_paired = results_df.sort_values(["dataset", "competitor", "split", "model"])
_print_block(_paired, "PAIRED  t2pmhc-GCN / t2pmhc-GAT  vs  competitor   (sorted by dataset, competitor)")

# Headline unseen set called out separately.
_igg = results_df[results_df.dataset == "iggytop-unseen"].sort_values(["test_kind", "competitor", "split", "model"])
_print_block(_igg, "iggytop-unseen  —  the only well-powered UNSEEN set (headline generalisation claim)")

n_up = int(results_df["underpowered"].sum())
print(f"\nunderpowered (N < 10) comparisons: {n_up} / {len(results_df)}")


# ## Summary tables

In [ ]:
# === Summary 1: significance vs statistical power, per (dataset, split, model) ===
# Paired GCN/GAT-vs-competitor tests only. Explains why the small-N Seen panels look
# 'null': nothing survives Holm at low N, while the effect is unambiguous on the
# well-powered iggytop-unseen set. sig_* = count of competitors beaten at p < 0.05.
_pr = results_df[results_df.test_kind == "paired_vs_competitor"]
power_table = (_pr.groupby(["dataset", "split", "model"])
               .agg(n_comparisons=("N", "size"),
                    median_N=("N", "median"),
                    sig_raw=("p_raw", lambda s: int((s < 0.05).sum())),
                    sig_holm=("p_holm", lambda s: int((s < 0.05).sum())))
               .reset_index()
               .sort_values(["dataset", "split", "model"]))
power_table["median_N"] = power_table["median_N"].astype(int)
power_table.to_csv(f"{RESULTS_DIR}/significance_power_summary.csv", index=False)
print("Paired GCN/GAT vs competitor — significant competitors per cell "
      "(sig / n_comparisons); Holm within each (model, dataset, split) family:")
power_table

In [ ]:
# === Summary 2: conclusive table — iggytop-unseen (the well-powered UNSEEN set) ===
# Paired two-sided Wilcoxon signed-rank, t2pmhc-GCN / t2pmhc-GAT vs each competitor, on
# the shared leakage-free per-peptide AUC pairs; Holm-corrected within each model family.
# This is the headline unseen-generalisation result.
_ig = (results_df[(results_df.test_kind == "paired_vs_competitor") &
                  (results_df.dataset == "iggytop-unseen")]
       .sort_values(["model", "competitor"]))
iggytop_table = (_ig[["model", "competitor", "N",
                      "median_model", "median_reference", "median_diff",
                      "ci_low", "ci_high", "statistic",
                      "p_raw", "p_holm", "rank_biserial"]]
                 .rename(columns={"median_reference": "median_competitor",
                                  "ci_low": "diff_ci_low", "ci_high": "diff_ci_high"})
                 .reset_index(drop=True))
iggytop_table["sig_holm"] = iggytop_table["p_holm"] < 0.05
iggytop_table.to_csv(f"{RESULTS_DIR}/iggytop_unseen_conclusive.csv", index=False)

_ng = int(iggytop_table[iggytop_table.model == "t2pmhc-GCN"]["sig_holm"].sum())
_na = int(iggytop_table[iggytop_table.model == "t2pmhc-GAT"]["sig_holm"].sum())
_nc = iggytop_table["competitor"].nunique()
print(f"iggytop-unseen (paired, Holm-corrected): "
      f"t2pmhc-GCN significantly beats {_ng}/{_nc} competitors; t2pmhc-GAT beats {_na}/{_nc}. "
      f"Positive rank-biserial / median_diff => t2pmhc higher.")
iggytop_table.round(4)